# 01 — Enriquecimento revisado de gênero

Prepara casos desconhecidos, pesquisa evidências públicas e publica somente candidatos aprovados.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
REPO_DIR = Path("/content/falando_nela")
REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_REF = ""  # Opcional: branch, tag ou commit; vazio acompanha o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

os.chdir(REPO_DIR)
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-analise.txt"], check=True)
print("Data root:", DATA_ROOT)
print("Commit:", subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())

## Configuração

Use o mesmo `RUN_ID` em toda a suíte. A configuração versionada é a fonte de verdade.

In [ ]:
from analise.discursos_plenario.config import load_config, resolve_input_paths, resolve_output_root

RUN_ID = "analise-plenario-20260713-v1"
CONFIG_PATH = REPO_DIR / "analise" / "discursos_plenario" / "config.v1.json"
ANALYSIS_CONFIG = load_config(CONFIG_PATH)
RUN_OUTPUT_ROOT = resolve_output_root(ANALYSIS_CONFIG, DATA_ROOT, RUN_ID)
INPUT_PATHS = resolve_input_paths(ANALYSIS_CONFIG, DATA_ROOT)
RODAR_ETAPA = False

assert ANALYSIS_CONFIG.date_start == "2010-02-02"
assert ANALYSIS_CONFIG.date_end == "2026-07-13"
assert ANALYSIS_CONFIG.raw["complete_year_end"] == 2025
assert ANALYSIS_CONFIG.raw["ytd_year"] == 2026
print("Run:", RUN_ID)
print("Saida:", RUN_OUTPUT_ROOT)

## Decisão metodológica

A informação oficial nunca é alterada. Nome, foto, aparência e tratamento isolados são evidências inválidas; toda candidatura identificada requer revisão humana.

In [ ]:
GENERO_PERIODS_PATH = INPUT_PATHS["parliamentarian_periods"]
assert GENERO_PERIODS_PATH.exists(), GENERO_PERIODS_PATH
GENERO_RESEARCH_MODEL = ANALYSIS_CONFIG.raw["openai"]["gender_research_model"]
RODAR_PESQUISA_WEB = False
PUBLICAR_REVISAO = False
MAX_CASOS_PESQUISA = None
print("Modelo de pesquisa:", GENERO_RESEARCH_MODEL)

## Execução

A etapa cara permanece desativada até a inspeção das entradas e dos parâmetros acima.

In [ ]:
from analise.discursos_plenario.genero import run_gender_enrichment_setup

GENERO_SETUP_RESULT = None
if RODAR_ETAPA:
    GENERO_SETUP_RESULT = run_gender_enrichment_setup(
        data_root=DATA_ROOT,
        run_id=RUN_ID,
        config_path=CONFIG_PATH,
        overwrite=False,
    )
    print(GENERO_SETUP_RESULT["manifest_path"])
else:
    print("Fila não gerada. Defina RODAR_ETAPA=True após revisar o contrato.")

## Validação imediata

Esta checagem não substitui os testes sintéticos nem a revisão dos manifests.

In [ ]:
import pandas as pd

GENERO_UNKNOWN_PATH = RUN_OUTPUT_ROOT / "01_genero" / "parlamentares_genero_desconhecido.csv"
if GENERO_UNKNOWN_PATH.exists():
    GENERO_UNKNOWNS = pd.read_csv(GENERO_UNKNOWN_PATH)
    assert GENERO_UNKNOWNS["parlamentar_key"].is_unique
    display(GENERO_UNKNOWNS.head())

## Pesquisa pública opcional

A célula abaixo usa a chave disponível no ambiente ou nos Secrets do Colab. Ela não imprime nem persiste a chave. Resultados continuam pendentes até revisão.

In [ ]:
import os
import pandas as pd
from openai import OpenAI
from analise.discursos_plenario.genero import research_gender_candidates
from analise.discursos_plenario.io import write_dataframe_atomic

GENERO_RESEARCH_RESULT = None
if RODAR_PESQUISA_WEB:
    if not os.environ.get("OPENAI_API_KEY"):
        try:
            from google.colab import userdata
            GENERO_SECRET = userdata.get("OPENAI_API_KEY")
        except Exception:
            GENERO_SECRET = None
        if GENERO_SECRET:
            os.environ["OPENAI_API_KEY"] = GENERO_SECRET
    assert os.environ.get("OPENAI_API_KEY"), "Configure OPENAI_API_KEY no ambiente ou nos Secrets do Colab."
    GENERO_CLIENT = OpenAI()
    GENERO_UNKNOWNS_FOR_RESEARCH = pd.read_csv(GENERO_UNKNOWN_PATH)
    GENERO_CANDIDATES, GENERO_ERRORS = research_gender_candidates(
        GENERO_UNKNOWNS_FOR_RESEARCH,
        client=GENERO_CLIENT,
        model=GENERO_RESEARCH_MODEL,
        prompt_version=ANALYSIS_CONFIG.raw["openai"]["gender_prompt_version"],
        limit=MAX_CASOS_PESQUISA,
    )
    write_dataframe_atomic(GENERO_CANDIDATES, RUN_OUTPUT_ROOT / "01_genero" / "revisao_genero.csv")
    write_dataframe_atomic(GENERO_ERRORS, RUN_OUTPUT_ROOT / "01_genero" / "erros_pesquisa_genero.csv")
    GENERO_RESEARCH_RESULT = {"candidatos": len(GENERO_CANDIDATES), "erros": len(GENERO_ERRORS)}
    print(GENERO_RESEARCH_RESULT)
else:
    print("Pesquisa web desativada.")

## Publicação após revisão humana

Edite `revisao_genero.csv` no Drive e preencha status, revisor e data. A função rejeita linhas incompletas.

In [ ]:
from analise.discursos_plenario.genero import publish_gender_review

GENERO_PUBLICATION_RESULT = None
GENERO_REVIEW_PATH = RUN_OUTPUT_ROOT / "01_genero" / "revisao_genero.csv"
if PUBLICAR_REVISAO:
    GENERO_PUBLICATION_RESULT = publish_gender_review(
        data_root=DATA_ROOT,
        run_id=RUN_ID,
        review_path=GENERO_REVIEW_PATH,
        config_path=CONFIG_PATH,
    )
    print(GENERO_PUBLICATION_RESULT["manifest_path"])
else:
    print("Publicação desativada.")